# Create Mini HDF5 Cache

Create one shared 20k train / 10k val cache from the full CLIP HDF5 cache on Google Drive. The mini H5 files are written on Colab local disk first and then copied to Drive.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure cache paths

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/blip2_project")
FULL_CACHE_DIR = DATA_ROOT / "cache"
MINI_CACHE_DIR = DATA_ROOT / "cache_mini_20k_10k"
LOCAL_WORK_DIR = Path("/content/mini_h5_cache")
SEED = 20260522
SPLIT_SIZES = {"train": 20000, "val": 10000}

FULL_H5 = {split: FULL_CACHE_DIR / f"{split}_features.h5" for split in SPLIT_SIZES}
MINI_H5 = {split: MINI_CACHE_DIR / f"{split}_features.h5" for split in SPLIT_SIZES}
MANIFEST_PATH = MINI_CACHE_DIR / "mini_cache_manifest.json"
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
MINI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Full cache:", FULL_CACHE_DIR)
print("Mini cache:", MINI_CACHE_DIR)
print("Local work:", LOCAL_WORK_DIR)

## 3. Build helpers

In [ ]:
import json
import os
import random
import h5py
from tqdm.auto import tqdm

def selected_keys(source_h5, size, seed):
    with h5py.File(source_h5, "r") as src:
        keys = sorted(src.keys(), key=int)
    if len(keys) < size:
        raise ValueError(f"{source_h5} has {len(keys)} keys, need {size}")
    rng = random.Random(seed)
    return sorted(rng.sample(keys, size), key=int)

def inspect_h5(path, expected_keys):
    with h5py.File(path, "r") as h5:
        keys = set(h5.keys())
        if keys != set(expected_keys):
            raise ValueError(f"Key mismatch for {path}: got {len(keys)}, expected {len(expected_keys)}")
        sample_key = expected_keys[0]
        sample = h5[sample_key]
        return {"count": len(keys), "sample_key": sample_key, "shape": list(sample.shape), "dtype": str(sample.dtype)}

def copy_split_local(source_h5, local_h5, keys):
    if local_h5.exists():
        local_h5.unlink()
    with h5py.File(source_h5, "r") as src, h5py.File(local_h5, "w") as dst:
        for key in tqdm(keys, desc=f"Copy {local_h5.stem}"):
            src.copy(key, dst, name=key)
    return inspect_h5(local_h5, keys)

def copy_file_to_drive(local_path, drive_path):
    tmp_path = drive_path.with_name(drive_path.name + ".tmp_sync")
    if tmp_path.exists():
        tmp_path.unlink()
    chunk_size = 64 * 1024 * 1024
    total_bytes = local_path.stat().st_size
    with open(local_path, "rb") as src, open(tmp_path, "wb") as dst:
        with tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024, desc=f"Sync {drive_path.name}") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))
        dst.flush()
        os.fsync(dst.fileno())
    if tmp_path.stat().st_size != local_path.stat().st_size:
        raise IOError(f"Drive temp copy size mismatch for {drive_path}")
    os.replace(tmp_path, drive_path)

def write_json_atomic(path, payload):
    tmp_path = path.with_name(path.name + ".tmp_sync")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp_path, path)

def existing_cache_is_valid(manifest):
    if manifest.get("seed") != SEED or manifest.get("split_sizes") != SPLIT_SIZES:
        return False
    for split, output_h5 in MINI_H5.items():
        if not output_h5.exists():
            return False
        try:
            inspect_h5(output_h5, manifest["selected_image_ids"][split])
        except (KeyError, OSError, ValueError):
            return False
    return True

## 4. Create or verify mini cache

In [ ]:
for path in FULL_H5.values():
    if not path.exists():
        raise FileNotFoundError(f"Missing full H5 cache: {path}")

manifest = None
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    if existing_cache_is_valid(manifest):
        print("Mini cache already valid. No files were rebuilt.")

if manifest is None or not existing_cache_is_valid(manifest):
    selected = {
        split: selected_keys(FULL_H5[split], size, SEED + index)
        for index, (split, size) in enumerate(SPLIT_SIZES.items())
    }
    summaries = {}
    for split, keys in selected.items():
        print(f"Build {split}: {len(keys):,} images")
        local_h5 = LOCAL_WORK_DIR / f"{split}_features.h5"
        summaries[split] = copy_split_local(FULL_H5[split], local_h5, keys)
        copy_file_to_drive(local_h5, MINI_H5[split])
        inspect_h5(MINI_H5[split], keys)
    manifest = {
        "seed": SEED,
        "split_sizes": SPLIT_SIZES,
        "source_h5": {split: str(path) for split, path in FULL_H5.items()},
        "output_h5": {split: str(path) for split, path in MINI_H5.items()},
        "selected_image_ids": selected,
        "summaries": summaries,
    }
    write_json_atomic(MANIFEST_PATH, manifest)
    print("Mini cache created:", MANIFEST_PATH)

for split, summary in manifest["summaries"].items():
    print(split, summary)